In [0]:
%%configure -n project.spark.compatibility
{
    "number_of_workers": 10,
    "session_type": "etl",
    "glue_version": "5.1",
    "worker_type": "G.1X",
    "idle_timeout": 15,
    "timeout": 60,
    "--enable-glue-datacatalog": "true",
    "--enable-auto-scaling": "true"
}


In [0]:
%%pyspark project.spark.compatibility
import sys
from pyspark.context import SparkContext
from pyspark.sql import SparkSession

import json
import boto3
import logging
from typing import Optional
from awsglue.utils import getResolvedOptions
from pyspark.sql.functions import *
from awsglue.context import GlueContext
from awsglue.job import Job


In [0]:
%%pyspark project.spark.compatibility
sc = SparkContext.getOrCreate()
spark = SparkSession.builder.getOrCreate()
glueContext = GlueContext(sc)
glue = glueContext  # alias used by some original Glue scripts


In [0]:
%%pyspark project.spark.compatibility
# Job parameters - modify as needed
glue_connection_jdbc = "smus-seed-jdbc-conn"
data_bucket = "smus-seed-glue-data-571600872292-us-east-1"
rds_database = "seeddb"
catalog_db_curated = "smus-seed-db-curated"
customers_table = "smus_seed_customers_parquet"
products_table = "smus_seed_products_parquet"

# Bundle into args dict for getResolvedOptions-style scripts
args = {
    "glue_connection_jdbc": glue_connection_jdbc,
    "data_bucket": data_bucket,
    "rds_database": rds_database,
    "catalog_db_curated": catalog_db_curated,
    "customers_table": customers_table,
    "products_table": products_table,
}


In [0]:
%%pyspark project.spark.compatibility
"""Seed Glue ETL job — RDS Postgres → S3 curated Parquet.

This is the script body uploaded to ``s3://<data-bucket>/scripts/<prefix>-rds-to-parquet.py``
by ``seed/glue/create.sh`` and registered as the ``<prefix>-rds-to-parquet``
Glue job (glueetl, Glue version 4.0). The job reads the two seed RDS
tables (``customers`` and ``products``) over the seed JDBC Glue
connection and writes them as Parquet partitions into the curated zone
of the seed data bucket.

Outputs:
    s3://<data-bucket>/curated/customers/   -- Parquet
    s3://<data-bucket>/curated/products/    -- Parquet

The Glue catalog tables ``<prefix>_customers_parquet`` and
``<prefix>_products_parquet`` are pre-registered (empty) in the
``<prefix>-db-curated`` database by ``seed/glue/create.sh``. This job is
what populates them.

Job parameters (passed from create.sh via ``--default-arguments``):
    --JOB_NAME            (set automatically by Glue)
    --glue_connection_jdbc <prefix>-jdbc-conn
    --data_bucket         <prefix>-glue-data-<account>-<region>
    --rds_database        seeddb (the JDBC connection's database name)
    --catalog_db_curated  <prefix>-db-curated
    --customers_table     <prefix>_customers_parquet
    --products_table      <prefix>_products_parquet
"""

# pylint: disable=import-error
import sys

from awsglue.dynamicframe import DynamicFrame

def _read_table(glue, args, table_name):
    """Read a single Postgres table over the seed JDBC connection.

    Returns a Glue DynamicFrame. The Glue connection's
    ``JDBC_CONNECTION_URL`` already names the database, so we only need
    to specify ``dbtable`` here.
    """
    return glueContext.create_dynamic_frame.from_options(
        connection_type="postgresql",
        connection_options={
            "useConnectionProperties": "true",
            "connectionName": glue_connection_jdbc,
            "dbtable": f"public.{table_name}",
        },
        transformation_ctx=f"read_{table_name}",
    )

def _write_parquet(glue, dyf, args, prefix, db_table_name):
    """Write a DynamicFrame as Parquet under ``s3://<bucket>/curated/<prefix>/``.

    Updates the matching Glue catalog table on every run so a schema
    evolution in the source RDS automatically lands as a catalog
    update; without this, downstream queries against the curated table
    would see stale column metadata.
    """
    target = f"s3://{data_bucket}/curated/{prefix}/"
    glueContext.write_dynamic_frame.from_options(
        frame=dyf,
        connection_type="s3",
        format="parquet",
        connection_options={"path": target},
        format_options={"compression": "snappy"},
        transformation_ctx=f"write_{prefix}",
    )

customers_dyf = _read_table(glue, args, "customers")
products_dyf = _read_table(glue, args, "products")

_write_parquet(glue, customers_dyf, args, "customers", customers_table)
_write_parquet(glue, products_dyf, args, "products", products_table)
